In [22]:
import argparse
import os
import sys
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score # For clustering

import plotly.express as px

import json

In [2]:
INPUT_DIR = "../../data/embedded"
OUTPUT_DIR = "../../data/results"
GENERATE_PLOT = True
ALGORITHM = "dbscan"
N_CLUSTERS = 5
EPS = 10.0
MIN_SAMPLES = 10
REDUCER = "pca"
PERPLEXITY = 30.0

In [3]:
# LOAD DATA
def load_data_recursive(input_dir):
    embeddings_list = []
    filenames = []

    print(f"Scanning '{input_dir}' for embeddings...")

    files_found = 0
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".npy"):
                file_path = os.path.join(root, file)

                try:
                    data = np.load(file_path)

                    if data.ndim == 0:
                        continue
                    elif data.ndim == 2:
                        data = data.squeeze()

                    if data.shape != (1024,):
                        continue

                    embeddings_list.append(data)
                    filenames.append(file.replace(".npy", ""))
                    files_found += 1

                except Exception as e:
                    print(f"Error loading {file}: {e}")

    if files_found == 0:
        return None, None

    print(f"Found {files_found} valid files.")

    X = np.vstack(embeddings_list)

    if not os.path.exists(INPUT_DIR):
        print(f"Error: Directory '{INPUT_DIR}' not found.")
        sys.exit(0)
    
    if X is None:
        print("No valid embeddings found. Check your directory.")
        sys.exit(0)
    
    print(f"Final Matrix Shape: {X.shape} (Samples: {X.shape[0]}, Features: {X.shape[1]})")
    return X, filenames

In [7]:
# PREPROCESSING
def preprocess(data):
    print("Scaling features...")
    scaler = StandardScaler()
    return scaler.fit_transform(data)

In [8]:
# CLUSTERING KMEANS
def cluster_kmeans(data, n_clusters):
    print(f"Running Clustering (kmeans)...")
    print(f"KMeans with {n_clusters} clusters")
    
    model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    
    return model.fit_predict(data)

In [9]:
# CLUSTERING DBSCAN
def cluster_dbscan(data, eps, min_samples):
    print(f"Running Clustering (dbscan)...")
    print(f"DBSCAN with eps={eps}, min_samples={min_samples}")
    
    model = DBSCAN(eps=eps, min_samples=min_samples)
    
    return model.fit_predict(data)

In [10]:
# DIMENSIONALITY REDUCTION
def reduce_pca(data, n_components):
    print(f"Reducing dimensions using (PCA)...")

    reducer = PCA(n_components=n_components)
    return reducer.fit_transform(data)

In [11]:
def reduce_tsne(data, perplexity, n_components):
    pca_50 = PCA(n_components=min(50, X.shape[1]))
    X_pca = pca_50.fit_transform(data)

    tsne = TSNE(n_components=n_components, perplexity=perplexity, random_state=42, init='pca', learning_rate='auto')
    return tsne.fit_transform(X_pca)

In [26]:
# VISUALIZATION
def create_save_scatter_plot(data, labels, file_labels, true_labels_path = '../../data/preprocessed/cleaned_labels.csv', title = "MOMENT Embeddings: PCA/TSNE Projection<br>(KMEANS/DBSCAN)", output_dir = "../../data/resultsplot.html"):
    # CREATE OUTPUT DIR
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    df_plot = pd.DataFrame(data, columns=['Component 1', 'Component 2'])
    
    df_plot['Cluster'] = labels.astype(str) 
    df_plot['Source'] = file_labels
    
    df_plot['ecg_id'] = df_plot['Source'].astype(int)
    
    try:
        true_labels_df = pd.read_csv(true_labels_path)
        df_plot = df_plot.merge(true_labels_df[['ecg_id', 'label']], on='ecg_id', how='left')
        df_plot['label'] = df_plot['label'].fillna('Unknown')
    except Exception as e:
        print(f"Could not load true labels: {e}")
        df_plot['label'] = 'Unknown'
    
    print("Generating Interactive Plot...")

    fig = px.scatter(
        df_plot,
        x='Component 1',
        y='Component 2',
        color='label',
        symbol='Cluster',
        hover_name='Source',
        hover_data={'label': True, 'Cluster': True, 'Component 1': False, 'Component 2': False},
        title=title,
        labels={
            'Component 1': "PCA/TSNE Dimension 1",
            'Component 2': "PCA/TSNE Dimension 2"
        },
        color_discrete_sequence=px.colors.qualitative.Plotly
    )

    fig.update_traces(marker=dict(size=10, line=dict(width=1, color='DarkSlateGrey')), opacity=0.8)
    fig.update_layout(template="plotly_white")

    fig.write_html(output_dir)
    print(f"Interactive plot saved to: {output_dir}")

    return df_plot

In [32]:
# CLUSTERING EVALUATION
def evaluate_clustering(X_scaled, labels, df_plot):
    print("\n" + "="*40)
    print("CLUSTERING METRICS")
    print("="*40)
    
    # Silhouette Score
    sil_score = silhouette_score(X_scaled, labels)
    print(f"Silhouette Score: {sil_score:.4f} (Closer to 1 = better separated)")
    
    # External Metrics
    valid_idx = df_plot['label'] != 'Unknown'
    true_labels_valid = df_plot.loc[valid_idx, 'label']
    pred_clusters_valid = df_plot.loc[valid_idx, 'Cluster']
    
    if len(true_labels_valid) > 0:
        # Calculate ARI and NMI
        ari = adjusted_rand_score(true_labels_valid, pred_clusters_valid)
        nmi = normalized_mutual_info_score(true_labels_valid, pred_clusters_valid)
        
        print(f"Adjusted Rand Index (ARI): {ari:.4f} (Closer to 1 = better match to true labels)")
        print(f"Normalized Mutual Info (NMI):  {nmi:.4f} (Closer to 1 = more information shared)")
    else:
        print("Could not calculate ARI/NMI: No true labels found.")
    
    print("="*40 + "\n")

In [15]:
X, filenames = load_data_recursive(INPUT_DIR)

Scanning '../../data/embedded' for embeddings...
Found 16272 valid files.
Final Matrix Shape: (16272, 1024) (Samples: 16272, Features: 1024)


In [16]:
X = preprocess(X)

Scaling features...


In [21]:
def dbscan_eps_search(data):
    print("--- DBSCAN Hyperparameter Search (1024D) ---")

    test_eps_values = [5.0, 10.0, 15.0, 20.0, 25.0, 30.0, 35.0, 40.0, 45.0, 50.0]

    for test_eps in test_eps_values:
        model = DBSCAN(eps=test_eps, min_samples=10)
        labels = model.fit_predict(data)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)

        print(f"eps={test_eps:<5} | Clusters found: {n_clusters:<3} | Noise points: {n_noise}/{len(labels)}")

dbscan_eps_search(X)

--- DBSCAN Hyperparameter Search (1024D) ---
eps=5.0   | Clusters found: 0   | Noise points: 16272/16272
eps=10.0  | Clusters found: 0   | Noise points: 16272/16272
eps=15.0  | Clusters found: 0   | Noise points: 16272/16272
eps=20.0  | Clusters found: 4   | Noise points: 11100/16272
eps=25.0  | Clusters found: 2   | Noise points: 2040/16272
eps=30.0  | Clusters found: 1   | Noise points: 224/16272
eps=35.0  | Clusters found: 1   | Noise points: 33/16272
eps=40.0  | Clusters found: 1   | Noise points: 16/16272
eps=45.0  | Clusters found: 1   | Noise points: 11/16272
eps=50.0  | Clusters found: 1   | Noise points: 5/16272


In [25]:
def dbscan_optimized_search(data):
    print("--- DBSCAN Hyperparameter Search (50D + L2) ---")

    test_eps_values = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]

    for test_eps in test_eps_values:
        model = DBSCAN(eps=test_eps, min_samples=10)
        labels = model.fit_predict(data)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)

        print(f"eps={test_eps:<5} | Clusters found: {n_clusters:<3} | Noise points: {n_noise}/{len(labels)}")

test_dimensions = [60, 55, 50, 45, 40, 35, 30, 25, 20, 15, 10]

for dims in test_dimensions:
    print(f"Current dimensions: {dims}")
    _X = reduce_pca(X, dims)
    _X = normalize(_X, norm="l2")
    dbscan_optimized_search(_X)

Current dimensions: 60
Reducing dimensions using (PCA)...
--- DBSCAN Hyperparameter Search (50D + L2) ---
eps=0.1   | Clusters found: 0   | Noise points: 16272/16272
eps=0.15  | Clusters found: 0   | Noise points: 16272/16272
eps=0.2   | Clusters found: 0   | Noise points: 16272/16272
eps=0.25  | Clusters found: 0   | Noise points: 16272/16272
eps=0.3   | Clusters found: 1   | Noise points: 16146/16272
eps=0.35  | Clusters found: 2   | Noise points: 15977/16272
eps=0.4   | Clusters found: 3   | Noise points: 15846/16272
eps=0.45  | Clusters found: 3   | Noise points: 15654/16272
eps=0.5   | Clusters found: 4   | Noise points: 15309/16272
Current dimensions: 55
Reducing dimensions using (PCA)...
--- DBSCAN Hyperparameter Search (50D + L2) ---
eps=0.1   | Clusters found: 0   | Noise points: 16272/16272
eps=0.15  | Clusters found: 0   | Noise points: 16272/16272
eps=0.2   | Clusters found: 0   | Noise points: 16272/16272
eps=0.25  | Clusters found: 0   | Noise points: 16272/16272
eps=0.3 

In [35]:
def run_pipeline(data, file_labels):
    print("--- REDUCING TO 10D ---")
    X_10d = reduce_pca(data, n_components=10)
    X_10d_norm = normalize(X_10d, norm="l2")

    print("--- RUNNING DBSCAN ---")
    labels = cluster_dbscan(X_10d_norm, eps=0.45, min_samples=10)
    print(pd.Series(labels).value_counts())

    print("--- GENERATING PLOT ---")
    X_2d = reduce_pca(X_10d_norm, n_components=2)
    df_plot = create_save_scatter_plot(X_2d, labels, file_labels, title="MOMENT Embeddings: DBSCAN (10D -> 2D)")

    print("--- CLUSTERING METRICS ---")
    valid_sil_idx = labels != -1
    evaluate_clustering(X_10d_norm[valid_sil_idx], labels[valid_sil_idx], df_plot)

run_pipeline(X, filenames)

--- REDUCING TO 10D ---
Reducing dimensions using (PCA)...
--- RUNNING DBSCAN ---
Running Clustering (dbscan)...
DBSCAN with eps=0.45, min_samples=10
 0    11911
-1     4336
 4        8
 2        7
 3        5
 1        5
Name: count, dtype: int64
--- GENERATING PLOT ---
Reducing dimensions using (PCA)...
Generating Interactive Plot...
Interactive plot saved to: ../../data/resultsplot.html
--- CLUSTERING METRICS ---

CLUSTERING METRICS
Silhouette Score: -0.1804 (Closer to 1 = better separated)
Adjusted Rand Index (ARI): 0.0114 (Closer to 1 = better match to true labels)
Normalized Mutual Info (NMI):  0.0020 (Closer to 1 = more information shared)

